# Tahap 1 Gambar 3.1 — Dataset CXR, Harmonisasi Multi-Sumber, Deduplikasi 2-Tahap, & Split 70/15/15

Notebook ini mengimplementasikan penyiapan data gabungan **Kermany Pediatric CXR + RSNA Pneumonia Detection Challenge** sesuai Proposal Bab III (§3.2.1 & Tabel 3.1):
1. **Filter & Harmonisasi Format Lossless:**
   - Kermany Mendeley (`5.856` JPEG) dikonversi eksplisit ke grayscale 1-channel.
   - RSNA Kaggle (`26.684` DICOM): agregasi baris multi-bbox ke tingkat citra unik, buang `11.821` kelas ambigu (`No Lung Opacity / Not Normal`), pertahankan `8.851` Normal + `6.012` Lung Opacity, balik polaritas `MONOCHROME1` menjadi `MONOCHROME2`, serta catat metadata `ViewPosition` (AP/PA) dan `PatientSex`.
   - Seluruh citra dari kedua sumber disimpan ke `data/interim/harmonized_png/` sebagai PNG grayscale 8-bit tanpa kompresi ulang (*lossless*).
2. **Deduplikasi 2-Tahap Sebelum Split:** SHA-256 byte/piksel + *perceptual hash* (`pHash` 64-bit, jarak Hamming $\le 5$) lintas sumber; berhenti keras bila ada konflik label.
3. **Penyeimbangan 1:1 per Sumber (`seed=42`):** Subsampling kelas mayoritas berbasis pasien pada masing-masing sumber (`Kermany 1.582:1.582 = 3.164`, `RSNA 6.012:6.012 = 12.024`, total target `15.188` citra) sehingga proporsi kelas seragam lintas sumber.
4. **Pembagian Data *Patient-Wise* 70:15:15 Terstratifikasi:** Grup = `group_id` (`kermany_<pid>` / `rsna_<patientId>`), strata = `sumber × kelas × posisi (AP/PA)`, dikunci dengan SHA-256 ke `data/interim/split_patientwise.csv`.

In [ ]:
# Sel 1: Setup lingkungan (Colab/Lokal), config terkunci, dan pemeriksaan direktori sumber
import json
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    !pip install -q pydicom

for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code"),
    Path("/content/gradcam-pneumonia"),
]:
    if (candidate / "src" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
from IPython.display import display

from src.data.dedup import run_deduplication
from src.data.manifest import build_kermany_manifest, build_multisource_manifest
from src.data.split import (
    patientwise_stratified_split,
    save_and_lock_split,
    source_balanced_subsample,
    verify_source_class_balance,
)
from src.paths import ensure_output_dirs, load_config
from src.seed import get_env_snapshot, print_env_banner, set_global_seed

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
SPLIT_SEED = int(CFG["assumed"]["split_seed"]["value"])
SUBSAMPLE_SEED = int(CFG["assumed"]["subsample_seed"]["value"])
PHASH_MAX_HAMMING = int(CFG["locked"]["data"]["phash_max_hamming"]["value"])

set_global_seed(SPLIT_SEED, deterministic=True)
print_env_banner(get_env_snapshot(seed=SPLIT_SEED))

KERMANY_DIR = PATHS["kermany_chest_xray"]
RSNA_DRIVE_DIR = PATHS["rsna_raw"]
WORK_RSNA_DIR = Path("/content/work/rsna")

# Gunakan SSD lokal Colab (/content/work/rsna) jika tersedia agar 10x lebih cepat dari FUSE Drive
if IN_COLAB and (WORK_RSNA_DIR / "stage_2_train_images").exists():
    RSNA_DIR = WORK_RSNA_DIR
elif IN_COLAB and (RSNA_DRIVE_DIR / "rsna-pneumonia-detection-challenge.zip").exists():
    import zipfile
    WORK_RSNA_DIR.mkdir(parents=True, exist_ok=True)
    needed = ("stage_2_train_labels.csv", "stage_2_detailed_class_info.csv", "stage_2_train_images/")
    with zipfile.ZipFile(RSNA_DRIVE_DIR / "rsna-pneumonia-detection-challenge.zip", "r") as zf:
        members = [m for m in zf.namelist() if m.startswith(needed)]
        print(f"Mengekstrak otomatis {len(members)} berkas RSNA dari ZIP Drive ke {WORK_RSNA_DIR} ...")
        zf.extractall(WORK_RSNA_DIR, members=members)
    RSNA_DIR = WORK_RSNA_DIR
else:
    RSNA_DIR = RSNA_DRIVE_DIR

# Di Colab, tulis PNG sementara di SSD lokal (/content/work/harmonized_png) agar konversi 20k gambar hanya ~2-3 menit
HARMONIZED_DIR = Path("/content/work/harmonized_png") if IN_COLAB else PATHS["harmonized_png"]
HARMONIZED_DIR.mkdir(parents=True, exist_ok=True)
HAS_RSNA = (RSNA_DIR / "stage_2_train_labels.csv").exists() and (RSNA_DIR / "stage_2_detailed_class_info.csv").exists()

print(f"\nIN_COLAB       : {IN_COLAB}")
print(f"PROJECT_ROOT   : {PATHS['project_root']}")
print(f"Kermany Raw    : {KERMANY_DIR} (ada={KERMANY_DIR.exists()})")
print(f"RSNA Raw Aktif : {RSNA_DIR} (ada={HAS_RSNA})")
print(f"Harmonized PNG : {HARMONIZED_DIR}")
if not HAS_RSNA:
    print(
        "\n[PERHATIAN] Dataset RSNA belum ditemukan di data/raw/rsna/!\n"
        "Letakkan berkas dari Kaggle 'rsna-pneumonia-detection-challenge' ke:\n"
        f"  {RSNA_DIR}/stage_2_train_labels.csv\n"
        f"  {RSNA_DIR}/stage_2_detailed_class_info.csv\n"
        f"  {RSNA_DIR}/stage_2_train_images/*.dcm"
    )

### Analisis Sel 1
Lingkungan eksekusi, seed deterministik, konfigurasi terkunci (`configs/config.yaml`), dan ketersediaan kedua sumber data (Kermany & RSNA) telah diperiksa.

In [ ]:
# Sel 2: Bangun manifest terpadu & harmonisasi format ke PNG grayscale 8-bit lossless
if HAS_RSNA:
    manifest_df, ingest_audit = build_multisource_manifest(
        kermany_root=KERMANY_DIR,
        rsna_root=RSNA_DIR,
        harmonized_dir=HARMONIZED_DIR,
        convert_to_png=True,
    )
    IMAGE_ROOT = HARMONIZED_DIR
    print("=== AUDIT INGESTI MULTI-SUMBER (KERMANY + RSNA) ===")
    print(json.dumps(ingest_audit, indent=2))
else:
    raise FileNotFoundError(
        f"Dataset RSNA wajib tersedia di {RSNA_DIR} sesuai proposal multi-sumber (§3.2.1). "
        "Unduh dari Kaggle 'rsna-pneumonia-detection-challenge' terlebih dahulu."
    )

manifest_path = PATHS["interim"] / "manifest_multisource_raw.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8")

print("\nDistribusi citra mentah per sumber x kelas:")
display(manifest_df.groupby(["source", "label"]).agg(n_images=("image_id", "count"), n_groups=("group_id", "nunique")).reset_index())
if "photometric" in manifest_df.columns:
    print("Sebaran PhotometricInterpretation (termasuk MONOCHROME1 yang telah dikoreksi):")
    display(manifest_df.groupby(["source", "photometric"]).size().reset_index(name="count"))
if "view_position" in manifest_df.columns:
    print("Sebaran ViewPosition (AP/PA) per sumber x kelas:")
    display(manifest_df.groupby(["source", "label", "view_position"]).size().reset_index(name="count"))

### Analisis Sel 2
Seluruh citra Kermany dan RSNA telah melalui satu alur konversi PNG grayscale 8-bit lossless, koreksi polaritas `MONOCHROME1`, serta pencatatan metadata proyeksi AP/PA dan jenis kelamin.

In [ ]:
# Sel 3: Deduplikasi 2-Tahap (SHA-256 byte+piksel & pHash 64-bit Hamming <= 5) lintas sumber
clean_df, dedup_log_df = run_deduplication(
    manifest_df=manifest_df,
    image_root=IMAGE_ROOT,
    check_pixel_hash=True,
    check_perceptual_hash=True,
    max_hamming=PHASH_MAX_HAMMING,
)

dedup_log_path = PATHS["interim"] / "dedup_log.csv"
dedup_log_df.to_csv(dedup_log_path, index=False, encoding="utf-8")

n_dropped = int((dedup_log_df["status"] == "DROPPED").sum())
print(f"Total citra pra-deduplikasi  : {len(manifest_df)}")
print(f"Total duplikat dikeluarkan   : {n_dropped}")
print(f"Total citra bersih           : {len(clean_df)}")
print("\nRincian duplikat menurut jenis kecocokan:")
display(dedup_log_df[dedup_log_df["status"] == "DROPPED"].groupby(["source", "label", "match_type"]).size().reset_index(name="n_dropped"))

### Analisis Sel 3
Deduplikasi 2-tahap memastikan tidak ada duplikat identik maupun citra hampir serupa (*near-duplicate*) lintas sumber yang bocor antar-partisi, dengan pengecekan ketat bebas konflik label.

In [ ]:
# Sel 4: Penyeimbangan 1:1 per Sumber (Patient-Wise) & Pembagian Data 70:15:15 Terkunci SHA-256
balanced_df, subsample_report_df = source_balanced_subsample(clean_df, seed=SUBSAMPLE_SEED)
balance_check = verify_source_class_balance(balanced_df, max_prevalence_gap=0.02)

subsample_report_csv = PATHS["outputs"] / "tables" / "eda" / "source_balancing_report.csv"
subsample_report_df.to_csv(subsample_report_csv, index=False, encoding="utf-8")
print("=== LAPORAN PENYEIMBANGAN 1:1 PER SUMBER ===")
display(subsample_report_df)
print(f"Verifikasi keseragaman proporsi kelas lintas sumber: {balance_check}")

split_df = patientwise_stratified_split(
    balanced_df,
    ratios=(
        float(CFG["locked"]["data"]["split_ratio_train"]["value"]),
        float(CFG["locked"]["data"]["split_ratio_val"]["value"]),
        float(CFG["locked"]["data"]["split_ratio_test"]["value"]),
    ),
    seed=SPLIT_SEED,
    stratify_by_view=True,
)

split_csv_path = PATHS["interim"] / "split_patientwise.csv"
checksum = save_and_lock_split(split_df, split_csv_path)

summary = split_df.groupby(["source", "split", "label"]).agg(
    n_images=("image_id", "count"),
    n_groups=("group_id", "nunique"),
).reset_index()
summary.to_csv(PATHS["outputs"] / "tables" / "eda" / "tabel_partisi_multisource.csv", index=False, encoding="utf-8")

print("\n===================================================================")
print("          GATE CHECK — NOTEBOOK 01 (DATA MULTI-SUMBER & SPLIT)     ")
print("===================================================================")
display(summary)
print(f"Total citra final terkunci : {len(split_df)}")
print(f"Berkas split terkunci      : {split_csv_path}")
print(f"SHA-256 Checksum           : {checksum}")
print("[PASS] 0% kebocoran pasien/citra antar-partisi & rasio kelas 1:1 per sumber terverifikasi.")